# LizyML Widget Tutorial

LizyML Widget is an interactive widget for operating LizyML features within Jupyter Notebook / Google Colab / VS Code Notebooks.

In this tutorial, you will learn:

1. **Loading Data** — Pass a DataFrame to the widget
2. **Data Tab** — Target selection, column settings, CV configuration
3. **Config Tab** — Model configuration, Fit / Tune execution
4. **Results Tab** — Metrics, plots, and parameter review
5. **Python API** — Programmatic operations (get config, inference, save)

## 0. Installation

```bash
pip install lizyml-widget lizyml[plots,tuning,calibration,explain]
```

> **Google Colab**: Prefix the command with `!`.  
> **Development mode**: Use `uv pip install -e ".[dev,lizyml]"` instead.

## 1. Preparing Data

First, let's create some sample data. Here we demonstrate a binary classification task.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)
n = 500

df = pd.DataFrame(
    {
        "id": range(1, n + 1),
        "age": rng.integers(18, 70, n),
        "income": rng.normal(50000, 15000, n).round(2),
        "category": rng.choice(["A", "B", "C"], n),
        "score": rng.uniform(0, 100, n).round(2),
        "constant_col": 1,  # auto-excluded (unique==1)
    }
)
# Binary target: income > 50000 with some noise
df["target"] = ((df["income"] > 50000) & (df["score"] > 30)).astype(int)

print(f"Shape: {df.shape}")
df.head()

## 2. Launching the Widget

Pass a DataFrame to `LizyWidget` and display it in a cell.  
Once the widget appears, the **Data tab** will be open.

In [ ]:
from lizyml_widget import LizyWidget

w = LizyWidget()
w.load(df)
w

## 3. Data Tab — Reviewing Auto-Detection

When data is loaded, the widget automatically detects the following:

| Item | Auto-Detection Logic |
|------|---------------------|
| **Task** | 2 unique values → binary / object or low unique count → multiclass / otherwise → regression |
| **ID column exclusion** | Columns where unique count == row count |
| **Constant column exclusion** | Columns where unique count == 1 |
| **Categorical columns** | dtype is object/category, or numeric columns with low unique count |
| **CV** | binary/multiclass → stratified_kfold / regression → kfold |

Select `target` in the Data tab of the widget displayed above, and these settings will be configured automatically.

You can also verify via the Python API:

In [ ]:
# Set target via Python API (can also be set from the widget UI)
w.set_target("target")

# Review auto-detected information
print(f"Task:     {w.task}")
print(f"CV:       {w.cv_method} ({w.cv_n_splits} folds)")
print(f"Shape:    {w.df_shape}")
print(f"Columns:  {w.df_columns}")

## 4. Config Tab — Model Configuration

The **Config tab** lets you edit LizyML settings via a GUI.  
Forms are dynamically generated from JSON Schema, so available parameters are displayed automatically.

You can also modify settings via the Python API:

In [ ]:
# Set model parameters via Python API
w.set_config(
    {
        "model": {
            "name": "lgbm",
            "params": {
                "n_estimators": 500,
                "learning_rate": 0.01,
                "max_depth": 5,
            },
        },
        "training": {
            "seed": 42,
            "early_stopping": {"enabled": True, "rounds": 100},
        },
    }
)

# Review current config
w.get_config()

## 5. Running Fit

Click the **Fit button** in the widget UI, or call `fit()` via the Python API.  
It runs in a background thread, and progress is displayed in the widget.

In [ ]:
# Run Fit via Python API
# (Same behavior as clicking the Fit button in the widget UI)
w.fit()

# Wait until the status changes from "fitting" to "done"
# The widget UI shows a real-time progress bar

## 6. Results Tab — Reviewing Results

After Fit completes, the **Results tab** displays metrics, plots, and parameters.  
You can also access results via the Python API.

In [ ]:
# Get the Fit result summary
summary = w.get_fit_summary()
if summary:
    print(f"Fold count: {summary.fold_count}")
    print(f"Metrics:    {summary.metrics}")
else:
    print("Fit has not been completed yet.")

In [ ]:
# Get the trained model object
model = w.get_model()
if model:
    print(type(model))
    # You can directly operate on the LizyML Model object
    # model.plot_learning_curve().show()
    # model.importance_plot(kind="split").show()

## 7. Tune — Hyperparameter Tuning

In the **Config tab's Tune sub-tab**, you can configure a search space and run hyperparameter tuning with Optuna.

In [ ]:
# Run Tune via Python API
# (Equivalent to clicking the Tune button in the widget UI)
w.tune()

# After completion, review the results
tune_summary = w.get_tune_summary()
if tune_summary:
    print(f"Best score:  {tune_summary.best_score:.4f}")
    print(f"Metric:      {tune_summary.metric_name} ({tune_summary.direction})")
    print(f"Best params: {tune_summary.best_params}")
    print(f"Trials:      {len(tune_summary.trials)}")

## 8. Inference — Predicting on New Data

Use the trained model to make predictions on new data.

In [ ]:
# Create new test data
df_test = pd.DataFrame(
    {
        "id": range(1001, 1011),
        "age": rng.integers(18, 70, 10),
        "income": rng.normal(50000, 15000, 10).round(2),
        "category": rng.choice(["A", "B", "C"], 10),
        "score": rng.uniform(0, 100, 10).round(2),
        "constant_col": 1,
    }
)

# Run inference via Python API
result = w.predict(df_test)
if result:
    print(f"Predictions shape: {result.predictions.shape}")
    print(f"Warnings: {result.warnings}")
    result.predictions.head()

## 9. Saving and Loading Config

You can save and load widget configuration as a YAML file.  
This is useful for sharing across teams or reproducing experiments.

In [ ]:
# Save config
w.save_config("my_experiment.yaml")
print("Config saved to my_experiment.yaml")

# To load config in another session:
# w2 = LizyWidget()
# w2.load(df)
# w2.load_config("my_experiment.yaml")

## 10. Regression Task Example

Simply changing the target column adapts the widget to a regression task.  
The task type is automatically detected.

In [ ]:
# Regression data
df_reg = pd.DataFrame(
    {
        "x1": rng.uniform(0, 10, 300),
        "x2": rng.uniform(-1, 1, 300),
        "x3": rng.choice(["low", "mid", "high"], 300),
    }
)
df_reg["price"] = df_reg["x1"] * 3.5 + df_reg["x2"] * 10 + rng.normal(0, 1, 300)

# Create and display the widget
w_reg = LizyWidget()
w_reg.load(df_reg)
w_reg.set_target("price")  # Continuous values → auto-detected as regression

print(f"Task: {w_reg.task}")  # → "regression"
print(f"CV:   {w_reg.cv_method}")  # → "kfold"
w_reg

## Python API Reference

| Method | Description |
|--------|-------------|
| `w.load(df)` | Load a DataFrame into the widget |
| `w.set_target(col)` | Set the target column (triggers auto-detection) |
| `w.set_config(dict)` | Set model configuration from Python |
| `w.get_config()` | Get the current configuration as a dict |
| `w.fit()` | Run Fit in the background |
| `w.tune()` | Run Tune in the background |
| `w.predict(df)` | Run inference on new data |
| `w.get_model()` | Get the LizyML Model object |
| `w.get_fit_summary()` | Get the Fit result as a FitSummary |
| `w.get_tune_summary()` | Get the Tune result as a TuningSummary |
| `w.save_config(path)` | Save config as YAML |
| `w.load_config(path)` | Load config from YAML |